# 💼 Skill Career Mapper
An AI agent that helps students understand **industry demand** for any technology skill and surfaces **real, matching job listings** from live job boards.

**Workflow:**
1. User provides a skill name
2. ReAct agent (Gemini 2.5 Flash) uses two tools:
   - `TavilySearch` (via MCP) → demand trends, salary ranges, career outlook
   - `search_jobs` → live job listings from JSearch (RapidAPI)
3. Returns a formatted report with job titles, companies, locations & apply links

## 📦 Section 1 — Install Dependencies

In [ ]:
!pip install -qU langchain langchain-google-genai langchain-tavily
!pip install -qU langchain-mcp-adapters requests

## 🔑 Section 2 — API Keys & Configuration
Add the following to **Google Colab Secrets** (🔑 icon in left sidebar):
- `GEMINI_API_KEY`
- `RAPIDAPI_KEY` — from https://rapidapi.com (subscribe to JSearch)
- `TAVILY_API_KEY`

> **MCP_TAVILY_URL** is the Composio-hosted Tavily MCP endpoint. Replace the placeholder with your own from https://composio.dev

In [ ]:
from google.colab import userdata

GEMINI_API_KEY  = userdata.get('GEMINI_API_KEY')
RAPIDAPI_KEY    = userdata.get('RAPIDAPI_KEY')
TAVILY_API_KEY  = userdata.get('TAVILY_API_KEY')

# ── Config ────────────────────────────────────────────────────────
LLM_MODEL      = 'google_genai:gemini-2.5-flash'
MCP_TAVILY_URL = 'https://backend.composio.dev/v3/mcp/<your-id>/mcp?user_id=<your-user-id>'

# Job search filters
JSEARCH_COUNTRY            = 'in'
JSEARCH_EMPLOYMENT_TYPES   = 'INTERN,FULLTIME'
JSEARCH_JOB_REQUIREMENTS   = 'no_experience,under_3_years_experience'

print('✅ Configuration loaded.')

## 🤖 Section 3 — Initialise LLM
We use **Gemini 2.5 Flash** via LangChain's unified `init_chat_model` interface.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(LLM_MODEL, api_key=GEMINI_API_KEY)

# Quick test
test = model.invoke('Reply with one word: ready')
print(f'✅ LLM ready. Test response: {test.content}')

## 🔌 Section 4 — Connect MCP Client (Composio Tavily)
The **MultiServerMCPClient** connects to the Composio-hosted Tavily MCP server and loads its tools so the agent can use them natively.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        'mcp_tavily': {
            'transport': 'http',
            'url': MCP_TAVILY_URL,
        }
    }
)

print('✅ MCP client configured.')

## 💼 Section 5 — Define Job Search Tool
The `search_jobs` tool calls the **JSearch API** via RapidAPI and returns structured job listings with title, company, location and a direct apply link.

In [ ]:
import requests
from langchain.tools import tool

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI.
    Returns a list of jobs with title, company, location, and apply link."""
    print(f'\n[search_jobs] Searching: "{skill}" in "{location}"')

    url = 'https://jsearch.p.rapidapi.com/search'
    headers = {
        'x-rapidapi-key': RAPIDAPI_KEY,
        'x-rapidapi-host': 'jsearch.p.rapidapi.com',
    }
    params = {
        'query': f'{skill} in {location}',
        'page': '1',
        'country': JSEARCH_COUNTRY,
        'employment_types': JSEARCH_EMPLOYMENT_TYPES,
        'job_requirements': JSEARCH_JOB_REQUIREMENTS,
    }

    try:
        response = requests.get(url, headers=headers, params=params, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        return [{'error': f'JSearch API error: {e}'}]

    jobs = response.json().get('data', [])
    print(f'[search_jobs] Found {len(jobs)} jobs.')

    return [
        {
            'title':      job.get('job_title'),
            'company':    job.get('employer_name'),
            'location':   job.get('job_city'),
            'apply_link': job.get('job_apply_link'),
        }
        for job in jobs
    ]

print('✅ search_jobs tool defined.')

## 🧩 Section 6 — Build the Agent
The agent is built **asynchronously** so it can fetch MCP tools at runtime and combine them with the local `search_jobs` tool. The system prompt instructs it to produce a clean, readable plain-text report.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are a Skill-to-Career Mapping assistant that helps students
understand skill demand and find matching job opportunities.

You have access to these tools:
- search tool (MCP Tavily): Search for industry demand, salary insights, and career trends.
- search_jobs: Find actual job listings requiring specific skills.

Help the student by researching the skill they ask about and finding relevant opportunities.
Present results in a clean, readable format with clear sections and proper spacing.
Include all job details with apply links. Do not use markdown formatting."""

async def build_agent():
    mcp_tools  = await client.get_tools()
    all_tools  = mcp_tools + [search_jobs]
    agent = create_agent(
        model=model,
        tools=all_tools,
        system_prompt=SYSTEM_PROMPT,
        debug=True,
    )
    return agent

print('✅ Agent builder defined.')

## 🚀 Section 7 — Run the Agent
Change `user_query` to ask about any skill or job market topic.

In [ ]:
user_query = 'What is the demand for generative AI in the industry?'

async def run_agent():
    agent    = await build_agent()
    response = await agent.ainvoke({
        'messages': [{'role': 'user', 'content': user_query}]
    })
    output = response['messages'][-1].content
    # Handle list-of-content-blocks response format
    if isinstance(output, list):
        output = '\n'.join(b.get('text', '') for b in output if isinstance(b, dict))
    print('\n' + '=' * 60)
    print(output)

print(f'Query: {user_query}\n')
await run_agent()

## 🔁 Section 8 — Try Different Skills *(Optional)*
Re-run this cell with different skill names to explore multiple career paths.

In [ ]:
queries = [
    'What is the demand for machine learning engineers in India?',
    'Find internships for Python developers in Bangalore',
    'What are the career prospects for full stack developers?',
]

# Change the index below to run a different query (0, 1, or 2)
selected_query = queries[0]

async def run_selected():
    agent    = await build_agent()
    response = await agent.ainvoke({
        'messages': [{'role': 'user', 'content': selected_query}]
    })
    output = response['messages'][-1].content
    if isinstance(output, list):
        output = '\n'.join(b.get('text', '') for b in output if isinstance(b, dict))
    print(f'Query: {selected_query}\n')
    print('=' * 60)
    print(output)

await run_selected()